# 07 | Kaggle predictions and submissions

Test feature generation and binary (0/1) Kaggle submissions, versions 2 and 3.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [ ]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)

import json
import joblib
test = load_events("test")
dataset_model = pd.read_parquet(data_path("dataset_model.parquet"))
models = joblib.load(artifact_path("model_specs.joblib"))  # uniquement un artefact local de confiance
with open(artifact_path("seuils.json"), encoding="utf-8") as f:
    seuils = json.load(f)
best_threshold = seuils["best_threshold"]
best_threshold_bal = seuils["best_threshold_bal"]


In [29]:

# ============================================
# FEATURES FOR KAGGLE TEST USERS
# ============================================

cutoff_test = pd.Timestamp("2018-11-20")
history_start = cutoff_test - HORIZON

# All test users
test_features = pd.DataFrame({
    "userId": test["userId"].unique()
})

# Events from the previous 10 days
history = test.loc[
    (test["time"] >= history_start) &
    (test["time"] < cutoff_test)
].copy()

# Event-level indicators
history["day"] = history["time"].dt.floor("D")
history["is_song"] = history["page"].eq("NextSong")
history["is_like"] = history["page"].eq("Thumbs Up")
history["is_dislike"] = history["page"].eq("Thumbs Down")
history["is_error"] = history["page"].eq("Error")
history["is_playlist"] = history["page"].eq("Add to Playlist")
history["is_downgrade"] = history["page"].eq("Submit Downgrade")

# Aggregations: the same features as in the training set
features = history.groupby("userId").agg(
    n_events_10d=("page", "size"),
    n_sessions_10d=("sessionId", "nunique"),
    n_active_days_10d=("day", "nunique"),
    n_songs_10d=("is_song", "sum"),
    n_likes_10d=("is_like", "sum"),
    n_dislikes_10d=("is_dislike", "sum"),
    n_errors_10d=("is_error", "sum"),
    n_playlist_adds_10d=("is_playlist", "sum"),
    total_song_length_10d=("length", "sum"),
    n_artists_10d=("artist", "nunique"),
    n_unique_songs_10d=("song", "nunique"),
    downgraded_10d=("is_downgrade", "max")
).reset_index()

test_features = test_features.merge(
    features,
    on="userId",
    how="left",
    validate="one_to_one"
)

# Users without recent activity
count_cols = features.columns.drop("userId")
test_features[count_cols] = test_features[count_cols].fillna(0)

# Subscription level before the prediction cutoff
past = test.loc[
    test["time"] < cutoff_test,
    ["userId", "time", "level"]
]

last_idx = past.groupby("userId")["time"].idxmax()

last_levels = (
    past.loc[last_idx]
    .set_index("userId")["level"]
)

test_features["is_paid"] = (
    test_features["userId"]
    .map(last_levels)
    .map({"paid": 1, "free": 0})
)

# Changes in song activity across two 10-day windows
previous_songs = test.loc[
    (test["time"] >= cutoff_test - 2 * HORIZON) &
    (test["time"] < history_start) &
    (test["page"] == "NextSong"),
    "userId"
].value_counts()

n_previous = (
    test_features["userId"]
    .map(previous_songs)
    .fillna(0)
)

n_recent = test_features["n_songs_10d"]

test_features["songs_trend_10d"] = (
    (n_recent - n_previous) /
    (n_recent + n_previous + 1)
)

# Sanity checks
print("Shape:", test_features.shape)

print("\nMissing values:")
print(test_features[FEATURES_12].isna().sum())

display(test_features[["userId"] + FEATURES_12].head())


Shape: (2904, 15)

Missing values:
n_sessions_10d           0
n_active_days_10d        0
n_songs_10d              0
n_likes_10d              0
n_dislikes_10d           0
n_errors_10d             0
n_playlist_adds_10d      0
total_song_length_10d    0
n_artists_10d            0
downgraded_10d           0
is_paid                  0
songs_trend_10d          0
dtype: int64


,userId,n_sessions_10d,n_active_days_10d,n_songs_10d,n_likes_10d,n_dislikes_10d,n_errors_10d,n_playlist_adds_10d,total_song_length_10d,n_artists_10d,downgraded_10d,is_paid,songs_trend_10d
0,1465194,6.0,5.0,287.0,12.0,3.0,2.0,8.0,70245.77345,262.0,False,1,0.134122
1,1261737,26442.0,10.0,0.0,0.0,0.0,147.0,0.0,0.00000,0.0,False,1,0.000000
2,1527155,1.0,2.0,301.0,13.0,7.0,0.0,5.0,76267.38353,280.0,False,1,0.996689
3,1507202,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0,1,0.000000
4,1429412,14.0,10.0,1839.0,100.0,17.0,2.0,50.0,453302.67623,1239.0,False,1,0.348607


In [32]:

from sklearn.base import clone

# ============================================
# 1. TRAIN ON ALL LABELED DATA
# ============================================

# Retrieve the logistic regression pipeline
model_final = clone(models["LogisticRegression"])

# Features and labels from all training windows
X_full = dataset_model[FEATURES_12]
y_full = dataset_model["churn"]

# Fit the model
model_final.fit(X_full, y_full)

print("Model trained on", len(X_full), "observations")


# ============================================
# 2. PREDICT FOR THE KAGGLE TEST SET
# ============================================

X_kaggle = test_features[FEATURES_12]

# Churn prediction scores
proba_kaggle = model_final.predict_proba(X_kaggle)[:, 1]

# Binary classification using the selected threshold
pred_kaggle = (proba_kaggle >= best_threshold).astype(int)


# ============================================
# 3. CREATE THE SUBMISSION FILE
# ============================================

# Create the Kaggle submission file
submission = pd.DataFrame({
    "id": test_features["userId"].values,
    "target": pred_kaggle.astype(int)
})

# Sanity checks
assert submission["id"].is_unique
assert submission["target"].isin([0, 1]).all()

print(submission.head(10))
print("\nPrediction distribution:")
print(submission["target"].value_counts())

# Export CSV
submission.to_csv("submission_churn_v2.csv", index=False)

print("\nFile created!")


Model trained on 60434 observations
        id  target
0  1465194       1
1  1261737       1
2  1527155       0
3  1507202       0
4  1429412       1
5  1778785       1
6  1776591       0
7  1937373       1
8  1959334       0
9  1138878       0

Prediction distribution:
target
0    2492
1     412
Name: count, dtype: int64

File created!


In [34]:

# Updated classification using the balanced-accuracy threshold
pred_kaggle_v3 = (
    proba_kaggle >= best_threshold_bal
).astype(int)

submission_v3 = pd.DataFrame({
    "id": test_features["userId"].values,
    "target": pred_kaggle_v3
})

submission_v3.to_csv(
    "submission_churn_v3.csv",
    index=False
)

print("Prediction distribution:")
print(submission_v3["target"].value_counts())

display(submission_v3.head())


Prediction distribution:
target
0    1745
1    1159
Name: count, dtype: int64


,id,target
0,1465194,1
1,1261737,1
2,1527155,0
3,1507202,0
4,1429412,1


In [ ]:
# The original cells generate CSV files in the project root.
# Also copy the files to submissions/ for a clear GitHub structure.
from shutil import copy2
for filename in ("submission_churn_v2.csv", "submission_churn_v3.csv"):
    if Path(filename).is_file():
        destination = Path("submissions") / filename
        copy2(filename, destination)
        print("Copied:", destination)